# ModelFlow — Results Viewer

Run **Setup** first, then run any layer cell to check your models.

In [2]:
import duckdb
import pandas as pd
from pathlib import Path
from IPython.display import display, Markdown

DB_PATH = Path("../data/modelflow.duckdb")
con = duckdb.connect(str(DB_PATH), read_only=True)

def show(table: str, limit: int = 5):
    """Find a table in any schema, print row count and a sample."""
    try:
        rows = con.execute(
            """SELECT table_schema
               FROM information_schema.tables
               WHERE table_name = ? AND table_type IN ('BASE TABLE', 'VIEW')
               LIMIT 1""",
            [table],
        ).fetchone()
    except Exception as e:
        display(Markdown(f"**`{table}`** — ❌ {e}"))
        return

    if rows is None:
        display(Markdown(f"**`{table}`** — 🔴 not built yet"))
        return

    schema = rows[0]
    count  = con.execute(f"SELECT COUNT(*) FROM {schema}.{table}").fetchone()[0]
    df     = con.execute(f"SELECT * FROM {schema}.{table} LIMIT {limit}").df()
    display(Markdown(f"**`{table}`** — `{schema}` — **{count:,} rows**"))
    display(df)


---

## Data Vault 2.0

In [3]:
## Bronze — staging + hash key computation
show("bronze_dv_stg_customers")
show("bronze_dv_stg_orders")
show("bronze_dv_stg_products")


**`bronze_dv_stg_customers`** — `main_dv_bronze` — **1,000 rows**

,cust_id,hk_customer,cust_name,cust_email,cust_country,cust_city,cust_segment,created_at,updated_at,hash_diff,load_date,record_source
0,1,c4ca4238a0b923820dcc509a6f75849b,Allison Hill,donaldgarcia@example.net,Uganda,New Roberttown,Home Office,2024-05-06,2026-02-27,751a066435dc3fc13bd6b64b811cef05,2026-04-13 11:35:57.967781,seeds.raw_customers
1,2,c81e728d9d4c2f636f067f89cc14862c,Kristina Baldwin,lrobinson@example.com,Sudan,Port Lindachester,Consumer,2024-11-02,2026-02-27,9e3daf497f91c91e77cf05a288ab6736,2026-04-13 11:35:57.967781,seeds.raw_customers
2,3,eccbc87e4b5ce2fe28308fd9f2a7baf3,Gabrielle Davis,howardmaurice@example.com,Sri Lanka,Lake Stephenville,Consumer,2026-02-07,2026-02-27,ed46aabe6e161225936ac69d6b68d1f0,2026-04-13 11:35:57.967781,seeds.raw_customers
3,4,a87ff679a2f3e71d9181a67b7542122c,Sandra Montgomery,barbara10@example.net,Denmark,Traciebury,Home Office,2025-11-21,2026-02-27,740d153f2bf6d9c1a964247ace5683a5,2026-04-13 11:35:57.967781,seeds.raw_customers
4,5,e4da3b7fbbce2345d7772b0674a318d5,Roy Martin,jason41@example.net,North Macedonia,New Jessica,Corporate,2024-06-26,2026-02-27,886815c9e077daabf377eefcc86ae501,2026-04-13 11:35:57.967781,seeds.raw_customers


**`bronze_dv_stg_orders`** — `main_dv_bronze` — **2,000 rows**

,order_id,cust_id,prod_id,hk_order,hk_customer,hk_product,hk_order_link,order_date,quantity,discount_pct,order_status,channel,payment_method,hash_diff,load_date,record_source
0,1,779,41,c4ca4238a0b923820dcc509a6f75849b,67d96d458abdef21792e6d8e590244e7,3416a75f4cea9109507cacd8e2f2aefc,64524126ca4d80e68e2cc003f569a90b,2025-04-17,2,5.0,Shipped,In-Store,Credit Card,77c871bbb15136ba448fe80db217c9e3,2026-04-13 11:35:57.982805,seeds.raw_orders
1,2,48,21,c81e728d9d4c2f636f067f89cc14862c,642e92efb79421734881b53e1e1b18b6,3c59dc048e8850243be8079a5c74d079,838391c87281ed4e5ffb6bcbcec939f7,2025-06-16,1,0.0,Cancelled,In-Store,Crypto,728c197a1273ddf3f407e1be28c1cc30,2026-04-13 11:35:57.982805,seeds.raw_orders
2,3,150,16,eccbc87e4b5ce2fe28308fd9f2a7baf3,7ef605fc8dba5425d6965fbd4c8fbe1f,c74d97b01eae257e44aa9d5bade97baf,58700139b5ae2a0020ef26238ab24dc5,2025-07-21,9,5.0,Returned,Mobile,PayPal,6c640e025c95d78c309a391abcccb454,2026-04-13 11:35:57.982805,seeds.raw_orders
3,4,180,6,a87ff679a2f3e71d9181a67b7542122c,045117b0e0a11a242b9765e79cbf113f,1679091c5a880faf6fb5e6087eb1b2dc,db1df942e230dc660213420301f97d71,2025-04-22,10,20.0,Pending,Mobile,Crypto,7eadc46b97bb326e78b9236a58c7deed,2026-04-13 11:35:57.982805,seeds.raw_orders
4,5,935,38,e4da3b7fbbce2345d7772b0674a318d5,e820a45f1dfc7b95282d10b6087e11c0,a5771bce93e200c36f7cd9dfd0e5deaa,8113e03247c6df9f7125c07990cbf3e3,2026-02-25,3,0.0,Pending,In-Store,Crypto,65c443ada5816d9e9dc3b97737e3c4e8,2026-04-13 11:35:57.982805,seeds.raw_orders


**`bronze_dv_stg_products`** — `main_dv_bronze` — **50 rows**

,prod_id,hk_product,product_name,category,subcategory,price,cost,sku,is_active,hash_diff,load_date,record_source
0,1,c4ca4238a0b923820dcc509a6f75849b,Customizable static neural-net,Sports,Attorney,194.64,93.31,PF-5352,True,18ff7dded8bda58c3a69cabfa1f61c79,2026-04-13 11:35:57.991482,seeds.raw_products
1,2,c81e728d9d4c2f636f067f89cc14862c,Ergonomic impactful analyzer,Sports,Help,104.39,136.82,HF-0791,True,c3f834bdd657545f4dcc17a8b44add39,2026-04-13 11:35:57.991482,seeds.raw_products
2,3,eccbc87e4b5ce2fe28308fd9f2a7baf3,Switchable solution-oriented Graphic Interface,Sports,Which,247.66,49.97,HZ-8998,True,4298df93b9a105157efb24493dc90bfc,2026-04-13 11:35:57.991482,seeds.raw_products
3,4,a87ff679a2f3e71d9181a67b7542122c,Versatile needs-based project,Sports,Deep,240.63,20.29,WM-4527,True,a1e5c2912cc6303aace720ed140e1423,2026-04-13 11:35:57.991482,seeds.raw_products
4,5,e4da3b7fbbce2345d7772b0674a318d5,Implemented responsive interface,Apparel,International,47.05,10.98,BF-6617,True,770c6d12f5f5f85e867400484ad737d7,2026-04-13 11:35:57.991482,seeds.raw_products


In [4]:
## Silver — Raw Vault (Hubs · Links · Satellites)
show("hub_customer")
show("hub_product")
show("link_order")
show("sat_customer_details")
show("sat_product_details")
show("sat_order_details")


**`hub_customer`** — `main_dv_silver` — **1,000 rows**

,hk_customer,cust_id,load_date,record_source
0,c16a5320fa475530d9583c34fd356ef5,31,2026-02-28 14:05:14.824011,seeds.raw_customers
1,e2c420d928d4bf8ce0ff2ec19b371514,71,2026-02-28 14:05:14.824011,seeds.raw_customers
2,e2ef524fbf3d9fe611d5a8e90fefdc9c,97,2026-02-28 14:05:14.824011,seeds.raw_customers
3,a3c65c2974270fd093ee8a9bf8ae7d0b,108,2026-02-28 14:05:14.824011,seeds.raw_customers
4,73278a4a86960eeb576a8fd4c9ec6997,113,2026-02-28 14:05:14.824011,seeds.raw_customers


**`hub_product`** — `main_dv_silver` — **50 rows**

,hk_product,prod_id,load_date,record_source
0,02e74f10e0327ad868d138f2b4fdd6f0,27,2026-02-28 14:05:14.897805,seeds.raw_products
1,c74d97b01eae257e44aa9d5bade97baf,16,2026-02-28 14:05:14.897805,seeds.raw_products
2,3c59dc048e8850243be8079a5c74d079,21,2026-02-28 14:05:14.897805,seeds.raw_products
3,a5bfc9e07964f8dddeb95fc584cd965d,37,2026-02-28 14:05:14.897805,seeds.raw_products
4,c16a5320fa475530d9583c34fd356ef5,31,2026-02-28 14:05:14.897805,seeds.raw_products


**`link_order`** — `main_dv_silver` — **2,000 rows**

...

,hk_customer,hash_diff,cust_name,cust_email,cust_country,cust_city,cust_segment,load_date,record_source
0,00ec53c4682d36f5c4359f4ae7bd7ba1,25e5207feaceddf3f732574ab26f5315,Bradley Shaw,kathleencastillo@example.org,Libyan Arab Jamahiriya,Pearsonborough,Corporate,2026-02-28 14:05:14.828739,seeds.raw_customers
1,04025959b191f8f9de3f924f0940515f,4529f68b154bf11560a82db8e68e7553,Debra Valentine,victorgonzalez@example.net,Dominican Republic,Jamieborough,Home Office,2026-02-28 14:05:14.828739,seeds.raw_customers
2,051e4e127b92f5d98d3c79b195f2b291,fb983841d3dcc5d24874d57f062c058d,Carolyn Smith,thompsondavid@example.net,India,North Emily,Corporate,2026-02-28 14:05:14.828739,seeds.raw_customers
3,0537fb40a68c18da59a35c2bfe1ca554,57f3eb30275ef9cf65986f1af99d34b8,Tracie Kim,jmorrow@example.net,Congo,Lake Elizabethberg,Consumer,2026-02-28 14:05:14.828739,seeds.raw_customers
4,0c74b7f78409a4022a2c4c5a5ca3ee19,c2f7483e54a88fd880ed0cc334d006c7,Andre Rodriguez,coreyduncan@example.com,Palestinian Territory,Foxshire,Home Office,2026-02-28 14:05:14.828739,seeds.raw_customers


**`sat_product_details`** — `main_dv_silver` — **50 rows**

,hk_product,hash_diff,product_name,category,subcategory,price,cost,sku,is_active,load_date,record_source
0,3c59dc048e8850243be8079a5c74d079,7422e8f5b909afca2cf59673e739460b,Enterprise-wide attitude-oriented task-force,Electronics,Rule,435.98,112.35,XC-1369,False,2026-02-28 14:05:14.905775,seeds.raw_products
1,f7177163c833dff4b38fc8d2872f1ec6,fd35526d1764450e2c3585a02327e115,Cross-group foreground help-desk,Electronics,Degree,443.39,143.29,XN-2341,False,2026-02-28 14:05:14.905775,seeds.raw_products
2,98f13708210194c475687be6106a3b84,31cbc0e11c5a2c908741cfbde82c7197,De-engineered multi-state alliance,Electronics,Manager,261.50,134.34,CY-7113,True,2026-02-28 14:05:14.905775,seeds.raw_products
3,a5771bce93e200c36f7cd9dfd0e5deaa,47637fb35eaa594f598ee622f38e318a,Extended hybrid circuit,Apparel,Many,72.52,45.65,XN-1874,True,2026-02-28 14:05:14.905775,seeds.raw_products
4,c20ad4d76fe97759aa27a0c99bff6710,42a1fa3bc6f439af0ec55972e4cd0dc4,Advanced 6thgeneration implementation,Home,Describe,186.61,149.77,QV-1403,True,2026-02-28 14:05:14.905775,seeds.raw_products


**`sat_order_details`** — `main_dv_silver` — **2,000 rows**

,hk_order_link,hash_diff,order_date,quantity,discount_pct,order_status,channel,payment_method,load_date,record_source
0,00f13c3be698b0bbbae8f139134de789,13200a80f7d878deabb676f2c1426146,2025-10-29,6,10.0,Cancelled,Partner,Bank Transfer,2026-02-28 14:05:14.891133,seeds.raw_orders
1,028b61183e1e76b3171961237d7a88d0,10f07f43bb6a044fdc39765a4f62610f,2025-11-29,5,15.0,Cancelled,Mobile,PayPal,2026-02-28 14:05:14.891133,seeds.raw_orders
2,031138203bc595a240735d601323170a,502f962f4f0e51c0e5da93de4e7b6936,2025-06-21,1,10.0,Pending,In-Store,PayPal,2026-02-28 14:05:14.891133,seeds.raw_orders
3,06664484af813ebf9311b1367e53877e,c79a9801455818e419abb7e5701db74c,2026-01-28,9,5.0,Shipped,Mobile,Bank Transfer,2026-02-28 14:05:14.891133,seeds.raw_orders
4,09d905d51a47ba024027555f8da1607d,8c2095cd6d78a7067548aa8337f1ff99,2025-05-30,8,0.0,Shipped,Mobile,Bank Transfer,2026-02-28 14:05:14.891133,seeds.raw_orders


In [5]:
## Gold — Business Vault
show("bv_customer_latest")
show("bv_sales_summary")


**`bv_customer_latest`** — `main_dv_gold` — **1,000 rows**

,hk_customer,cust_id,cust_name,cust_email,cust_country,cust_city,cust_segment,first_seen_date,last_updated_date,record_source
0,c16a5320fa475530d9583c34fd356ef5,31,Andrew Graham,michaeljones@example.net,Uruguay,Cindyville,Corporate,2026-02-28 14:05:14.824011,2026-02-28 14:05:14.828739,seeds.raw_customers
1,e2c420d928d4bf8ce0ff2ec19b371514,71,Mark Mccall,juliehamilton@example.org,Senegal,South Garrettport,Consumer,2026-02-28 14:05:14.824011,2026-02-28 14:05:14.828739,seeds.raw_customers
2,e2ef524fbf3d9fe611d5a8e90fefdc9c,97,James Padilla,charles06@example.org,Kuwait,Port Lawrencechester,Consumer,2026-02-28 14:05:14.824011,2026-02-28 14:05:14.828739,seeds.raw_customers
3,a3c65c2974270fd093ee8a9bf8ae7d0b,108,Zachary Moore,megan18@example.net,Syrian Arab Republic,Lake Matthewberg,Consumer,2026-02-28 14:05:14.824011,2026-02-28 14:05:14.828739,seeds.raw_customers
4,73278a4a86960eeb576a8fd4c9ec6997,113,Jonathan Young,brooksanthony@example.org,Taiwan,Laceyberg,Consumer,2026-02-28 14:05:14.824011,2026-02-28 14:05:14.828739,seeds.raw_customers


**`bv_sales_summary`** — `main_dv_gold` — **2,000 rows**

,order_id,hk_customer,hk_product,cust_name,cust_country,cust_segment,product_name,category,price,cost,order_date,quantity,discount_pct,order_status,channel,payment_method,gross_revenue,total_cost
0,12,c5ab0bc60ac7929182aadd08703f1ec6,70efdf2ec9b086079795c442636b55fb,Martin Hall,Georgia,Consumer,Face-to-face asynchronous productivity,Home,181.35,139.25,2025-03-29,3,15.0,Shipped,Web,Bank Transfer,462.44,417.75
1,53,35cf8659cfcb13224cbd47863a34fc58,a5bfc9e07964f8dddeb95fc584cd965d,Krystal Carter,Kyrgyz Republic,Home Office,Switchable zero tolerance process improvement,Home,223.29,152.05,2025-07-02,10,0.0,Shipped,Mobile,PayPal,2232.90,1520.50
2,58,9cfdf10e8fc047a44b08ed031e1f0ed1,4e732ced3463d06de0ca9a15b6153677,Ryan Wiley,Costa Rica,Corporate,Progressive executive open architecture,Sports,192.76,96.06,2025-06-04,10,0.0,Shipped,Mobile,Bank Transfer,1927.60,960.60
3,67,5807a685d1a9ab3b599035bc566ce2b9,b6d767d2f8ed5d21a44b0e5886680cb9,Jacqueline Chen,El Salvador,Corporate,User-centric client-server intranet,Home,205.52,76.88,2025-12-19,4,0.0,Shipped,Web,Credit Card,822.08,307.52
4,97,6da9003b743b65f4c0ccd295cc484e57,c81e728d9d4c2f636f067f89cc14862c,Derek Wilson,Qatar,Corporate,Ergonomic impactful analyzer,Sports,104.39,136.82,2025-03-17,6,0.0,Cancelled,Mobile,PayPal,626.34,820.92


In [6]:
## BONUS: Python Models
show("py_bv_customer_rfm")

**`py_bv_customer_rfm`** — 🔴 not built yet

---

## Inmon 3NF (CDW)

In [ ]:
## Bronze — staging (no hash keys)
show("bronze_3nf_stg_customers")
show("bronze_3nf_stg_orders")
show("bronze_3nf_stg_products")


In [ ]:
## Silver — 3NF CDW entities
show("dim_customer_3nf")
show("dim_product_3nf")
show("fact_order_3nf")


In [ ]:
## Gold — reporting aggregations
show("rpt_sales_by_customer")
show("rpt_sales_by_product")


In [7]:
## BONUS: Python Models
show("py_rpt_customer_ltv")

**`py_rpt_customer_ltv`** — 🔴 not built yet

---

## Star Schema

In [ ]:
## Bronze — staging + denormalised measures
show("bronze_ss_stg_customers")
show("bronze_ss_stg_orders")
show("bronze_ss_stg_products")


In [ ]:
## Silver — SCD2 dimensions + fact
show("dim_customer_scd2")
show("dim_product_scd2")
show("dim_date")
show("fact_sales")


In [ ]:
## Gold — BI marts
show("mart_sales_dashboard")
show("mart_product_performance")


In [8]:
## BONUS: Python Models
show("py_mart_sales_anomaly")

**`py_mart_sales_anomaly`** — 🔴 not built yet